---
title: 'Lab 8: Trening modeli gradientem prostym w PyTorch'
subtitle: Biblioteki Python w analizie danych
author: Tomasz Rodak
jupyter: python3
---


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rodakt/BPwAD/blob/v2/laby/lab_8.ipynb)

Na lab 7 stosowaliśmy metodę gradientu prostego do minimalizacji funkcji matematycznych — bez danych, bez modeli. Na wykładzie 4 zobaczyliśmy, że trening modeli uczenia maszynowego to w istocie **minimalizacja funkcji straty** i że w tej roli można wykorzystać dokładnie ten sam algorytm, tyle że zastosowany do funkcji zależnej od zbioru danych.

W tym arkuszu połączymy oba wątki: zastosujemy mini-batch gradient descent w PyTorch do treningu dwóch klasycznych modeli parametrycznych — regresji liniowej z nieliniowymi cechami (sekcja 2) oraz regresji logistycznej (sekcja 3). Gradienty będziemy nadal wyprowadzać i programować ręcznie; mechanizm różniczkowania automatycznego (*autograd*), który zdejmie z nas ten obowiązek, poznamy na wykładzie 5.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset, DataLoader

## 1. Rozgrzewka: tensory PyTorch

Na wykładzie 4 poznaliśmy tensory jako "numpy'owe tablice z dodatkową infrastrukturą". Zanim przejdziemy do treningu modeli, wykonajmy kilka krótkich zadań rozgrzewkowych, żeby nabrać wprawy w składni.

### 1.1 Tworzenie tensorów i typy danych

1. Utwórz tensor `a` z listy `[1.0, 2.0, 3.0, 4.0]`. Wypisz `a.shape`, `a.dtype` i `a.ndim`.
2. Utwórz tensor `b` z listy `[1, 2, 3, 4]` (bez kropek). Jaki jest jego `dtype`? Porównaj z poprzednim.
3. Utwórz tensor `z` wypełniony zerami o kształcie `(3, 4)` oraz tensor `r` o tym samym kształcie wypełniony liczbami z rozkładu $\mathcal{N}(0, 1)$.
4. Jakiego typu zmiennoprzecinkowego używa domyślnie PyTorch? Jakiego NumPy? Sprawdź eksperymentalnie.

### 1.2 NumPy ↔ PyTorch

1. Utwórz tablicę NumPy `arr = np.array([[1.0, 2.0], [3.0, 4.0]])`.
2. Przekształć ją na tensor dwoma sposobami: przez `torch.from_numpy(arr)` oraz przez `torch.tensor(arr)`.
3. Zmień jeden element oryginalnej tablicy NumPy: `arr[0, 0] = 99.0`. Który z dwóch tensorów "widzi" tę zmianę, a który nie? Wyjaśnij.

### 1.3 Operacje na tensorach

Niech $X \in \mathbb{R}^{5 \times 3}$ będzie losowym tensorem, a $w \in \mathbb{R}^3$ wektorem wag. Zaimplementuj następujące operacje (każdą jedną linijką):

1. Mnożenie macierzowe $Xw$ — wynik powinien być tensorem o kształcie `(5,)`.
2. Transpozycja: utwórz $X^T$ i sprawdź jej kształt.
3. Broadcasting: utwórz wektor $b \in \mathbb{R}^3$ i oblicz $X + b$ (każda kolumna $X$ zostaje przesunięta o odpowiadający element $b$).
4. Suma po osi: oblicz średnią $X$ po osi 0 (wynik kształtu `(3,)`) i po osi 1 (wynik kształtu `(5,)`)

### 1.4 `TensorDataset` i `DataLoader`

Wygeneruj sztuczny zbiór 50 próbek: $X \in \mathbb{R}^{50 \times 2}$ z rozkładu $\mathcal{N}(0, 1)$ oraz $y \in \mathbb{R}^{50}$ z rozkładu $\mathcal{N}(0, 1)$.

1. Opakuj dane w `TensorDataset`.
2. Utwórz `DataLoader` z `batch_size=8` i `shuffle=True`.
3. Przeiteruj po `DataLoader` **jedną** epokę. Dla każdego batcha wypisz jego numer i kształt `X_batch`, `y_batch`.
4. Ile batchy zostało zwróconych? Dlaczego właśnie tyle? Jakiego kształtu jest ostatni batch?

### 1.5 (Opcjonalnie) GPU

Sprawdź, czy w środowisku jest dostępny GPU:

```python
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
```

Jeśli tak, utwórz tensor `t = torch.randn(1000, 1000)` i przenieś go na GPU: `t = t.to(device)`. W Colab można włączyć GPU w `Środowisko wykonawcze → Zmień typ środowiska`.

## 2. Regresja liniowa z bazami gaussowskimi na plamach słonecznych

Zbiór danych **plam słonecznych** (SILSO, Królewskie Obserwatorium Belgii) jest klasycznym zbiorem szeregów czasowych: dzienna liczba plam na powierzchni Słońca mierzona od 1818 roku. Dane wykazują charakterystyczną cykliczność ~11-letnią, ale też zmienność amplitudy między cyklami (tzw. *cykl Gleissberga*).

Zbudujemy model, który odtworzy tę zależność za pomocą **regresji liniowej w bazie cech gaussowskich**. Model pozostaje liniowy w parametrach (co jest kluczowe — gradient ma prostą postać), ale dzięki nieliniowym cechom potrafi przybliżać zależności nieliniowe. To ten sam pomysł, który stoi za jądrowymi maszynami wektorów nośnych i, w pewnym sensie, za sieciami neuronowymi (warstwa ukryta to wyuczone, nieliniowe cechy).


### 2.1 Pobranie i przygotowanie danych

Plik CSV z SILSO jest dostępny pod adresem: 

`https://www.sidc.be/SILSO/INFO/sndtotcsv.php`

Ma 8 kolumn rozdzielonych **średnikiem**, bez nagłówka. Znaczenie kolumn (po kolei):

1. rok,
2. miesiąc,
3. dzień,
4. rok dziesiętny (ułamek roku),
5. **dzienna liczba plam słonecznych** (SN) — zmienna zależna,
6. odchylenie standardowe,
7. liczba obserwacji,
8. flaga definitywności (1 = dane ostateczne, 0 = prowizoryczne).

**Uwaga:** dni bez obserwacji są oznaczone wartością $-1$ w kolumnie SN — musimy je odfiltrować.

1. Załaduj dane funkcją `pd.read_csv` z parametrami `sep=';'`, `header=None`, własnymi `names` i `na_values=[-1]` dla kolumny SN.
2. Odfiltruj wiersze z brakującą wartością SN.
3. Zostaw dane od **1 stycznia 1874** (od tej daty obserwacje są praktycznie kompletne).
4. Wypisz: ile zostało rekordów, jaki jest zakres czasu (pierwsza i ostatnia data), minimum/maksimum/średnia SN.
5. Narysuj surowy wykres SN w funkcji roku dziesiętnego (pełen zakres). Seria jest mocno zaszumiona — to normalne. Opcjonalnie: nałóż wygładzoną ruchomą średnią z okna np. 365 dni, żeby wyraźniej zobaczyć cykle.

### 2.2 Standaryzacja czasu

Dla stabilności numerycznej przeskalujemy czas do przedziału $[0, 1]$:

$$\tau = \frac{t - t_{\min}}{t_{\max} - t_{\min}}$$

gdzie $t$ to rok dziesiętny (kolumna 4 z pliku).

1. Oblicz $\tau$ dla wszystkich punktów. Sprawdź, że $\tau \in [0, 1]$.
2. Wypisz $t_{\min}$ i $t_{\max}$ — przydadzą się w sekcji 2.10 do ekstrapolacji.

### 2.3 Bazy gaussowskie

Dla ustalonej liczby baz $K$ i szerokości $s > 0$ definiujemy $K$ funkcji bazowych:

$$\varphi_k(\tau) = \exp\!\left(-\frac{(\tau - \mu_k)^2}{2 s^2}\right), \qquad k = 1, 2, \ldots, K,$$

gdzie centra $\mu_k$ są równomiernie rozmieszczone na przedziale $[0, 1]$, np. $\mu_k = \frac{k - 1}{K - 1}$.

Każda baza jest "kapeluszem" Gaussa o maksimum w $\mu_k$ i szerokości kontrolowanej przez $s$. Model ma postać:

$$\hat{y}(\tau) = w_0 + \sum_{k=1}^K w_k \, \varphi_k(\tau),$$

czyli kombinację liniową baz plus wyraz wolny $w_0$. Dodanie kolumny jedynek do macierzy cech pozwala potraktować $w_0$ tak samo jak pozostałe wagi.


1. Napisz funkcję `gaussian_features(tau, K, s)` przyjmującą:
   - `tau` — tablicę NumPy lub tensor 1D kształtu `(N,)`,
   - `K` — liczbę baz,
   - `s` — szerokość bazy,
   
   i zwracającą tensor $\Phi \in \mathbb{R}^{N \times (K+1)}$, gdzie pierwsza kolumna to jedynki (*bias*), a kolumny $2, \ldots, K+1$ to wartości funkcji $\varphi_1, \ldots, \varphi_K$.
   
   *Wskazówka:* można użyć broadcastingu. Jeśli `tau` ma kształt `(N,)` i zmienisz go na `(N, 1)`, a centra `mu` na `(K,)`, to różnica `tau[:, None] - mu[None, :]` będzie kształtu `(N, K)`.

2. Dla $K = 10$ i $s = 0.05$ narysuj **wszystkie 10 baz** na jednym wykresie (oś $x$: $\tau \in [0, 1]$, oś $y$: $\varphi_k(\tau)$). Powinno powstać 10 "dzwonów" rozmieszczonych równomiernie.

### 2.4 Model, funkcja straty i gradient

Model: $\hat{\mathbf{y}} = \Phi \mathbf{w}$, gdzie $\Phi \in \mathbb{R}^{N \times (K+1)}$, $\mathbf{w} \in \mathbb{R}^{K+1}$.

Funkcja straty MSE:

$$L(\mathbf{w}) = \frac{1}{N} \|\mathbf{y} - \Phi \mathbf{w}\|^2.$$

Gradient (znany z wykładu 4, wyprowadzenie identyczne):

$$\nabla_{\mathbf{w}} L = -\frac{2}{N} \Phi^T (\mathbf{y} - \Phi \mathbf{w}).$$

Dla mini-batcha $B$ próbek wzór ma identyczną postać, z podmacierzami $\Phi_B$ i $\mathbf{y}_B$:

$$\nabla_{\mathbf{w}} L_B = -\frac{2}{B} \Phi_B^T (\mathbf{y}_B - \Phi_B \mathbf{w}).$$

### 2.5 Podział train/val i `DataLoader`

1. Ustal $K$ i $s$ — zacznij od $K = 200$, $s = 0.005$ (centra co $\approx 0.005$, szerokość porównywalna z odstępem).
2. Oblicz $\Phi$ dla wszystkich danych (kształt `(N, K+1)`) i tensor $\mathbf{y}$ (wartości SN jako `torch.float32`).
3. Dokonaj **losowego** podziału train/val 80/20. Możesz użyć `torch.randperm(N)` albo `sklearn.model_selection.train_test_split` na indeksach.
4. Utwórz `TensorDataset` i `DataLoader` dla zbioru treningowego z `batch_size=256`, `shuffle=True`. Zbiór walidacyjny trzymaj jako surowe tensory $\Phi_\text{val}$, $\mathbf{y}_\text{val}$ — nie iterujemy po nim w batchach, tylko liczymy stratę od razu na całości.

**Uwaga.** Skalowanie czasu do $[0, 1]$ wykonujemy na wszystkich danych jednocześnie, przed podziałem na train/val w sekcji 2.5. W odróżnieniu od `StandardScaler` w sekcji 3, gdzie statystyki (średnia, odchylenie standardowe) **uczymy się** ze zbioru treningowego, tutaj traktujemy $t_{\min}$ i $t_{\max}$ jako **zmianę jednostek**, a nie parametry modelu — zmieniamy oś czasu z lat na ułamki przedziału, analogicznie do zamiany stopni Celsjusza na Kelwiny.

### 2.6 Pętla treningowa

Zaimplementuj mini-batch GD:

1. Zainicjalizuj wagi $\mathbf{w}$ losowo: np. `torch.randn(K+1) * 0.01` (małe losowe wartości).
2. Ustal learning rate `lr` (np. 0.01) i liczbę epok (np. 30).
3. W każdej epoce:
   - iteruj po `DataLoader`ze;
   - dla każdego batcha oblicz $\hat{\mathbf{y}}_B = \Phi_B \mathbf{w}$, residua $\mathbf{r}_B = \mathbf{y}_B - \hat{\mathbf{y}}_B$, gradient według wzoru z 2.4;
   - aktualizuj wagi: $\mathbf{w} \gets \mathbf{w} - \eta \nabla L_B$.
4. Po każdej epoce oblicz **pełne** MSE na train i val (jednym mnożeniem macierzowym, bez iteracji po batchach) i dopisz do historii.
5. Wypisz co kilka epok aktualny train/val MSE.

*Wskazówka dot. `lr`:* jeśli MSE rośnie lub wybucha do NaN, learning rate jest za duży. Jeśli po 30 epokach train MSE prawie się nie zmienił, za mały. Eksperymentuj na zasadzie ×10 / ÷10.

### 2.7 Wizualizacja dopasowania

1. Po zakończeniu treningu oblicz predykcję modelu $\hat{\mathbf{y}} = \Phi \mathbf{w}$ na **całym** zakresie danych.
2. Narysuj na jednym wykresie:
   - surowe dane (punkty, alpha mały — np. 0.2, żeby zagęszczenie było widoczne);
   - predykcję modelu (linia, np. czerwona, o grubszej szerokości).
3. Czy model odtwarza cykliczność? Czy amplituda jest realistyczna?

Narysuj też krzywe uczenia: train MSE i val MSE w funkcji epoki na jednym wykresie (oś $y$ w skali logarytmicznej).

### 2.8 Sanity check z rozwiązaniem analitycznym (OLS)

Regresja liniowa ma rozwiązanie w postaci zamkniętej:

$$\mathbf{w}_{\text{OLS}} = (\Phi^T \Phi)^{-1} \Phi^T \mathbf{y}.$$

W praktyce używamy stabilniejszego numerycznie `np.linalg.lstsq` lub `torch.linalg.lstsq`.

1. Oblicz $\mathbf{w}_{\text{OLS}}$ na zbiorze treningowym.
2. Oblicz MSE modelu OLS na train i val.
3. Porównaj z wynikami GD po 30 epokach: wagi powinny być zbliżone, MSE też. Jeśli GD daje zauważalnie gorszy wynik, potrzeba więcej epok lub lepszego `lr`.

To ważny moment: **GD odtwarza rozwiązanie analityczne**. Dla regresji liniowej GD jest niepotrzebny (rozwiązanie zamknięte jest tańsze), ale ten sam algorytm będzie działał tam, gdzie zamkniętej formy już nie ma.

### 2.9 Eksperymenty z hiperparametrami

Najważniejsza część sekcji 2. Tu badamy, jak konstrukcja modelu i hiperparametry GD wpływają na wynik.

#### 2.9.1 Wpływ liczby baz $K$

Uruchom trening dla $K \in \{20, 200, 800\}$ przy stałym $s = 0.005$, `lr` i liczbie epok dobranych w 2.6.

Dla każdego $K$:

1. Narysuj dopasowanie modelu na tle danych (trzy osobne wykresy).
2. Zapisz końcowe wartości train MSE i val MSE.

Zbierz wyniki w tabeli. Od którego $K$ val MSE zaczyna rosnąć przy spadającym train MSE? Jak nazywamy to zjawisko?

#### 2.9.2 Wpływ szerokości bazy $s$

Ustaw $K = 200$. Uruchom trening dla trzech wartości $s$:

- $s = 0.001$ — bardzo wąskie bazy (bazy prawie nie nakładają się),
- $s = 0.005$ — standardowe (porównywalne z odstępem między centrami),
- $s = 0.05$ — szerokie (duże nakładanie).

Porównaj dopasowania na trzech wykresach. Który model za bardzo "dopasowuje się do szumu"? Który jest zbyt gładki?

#### 2.9.3 Wpływ `lr` i `batch_size`

Ustaw $K = 200$, $s = 0.005$. Uruchom trening dla siatki:

- `lr` $\in \{0.001, 0.01, 0.1\}$,
- `batch_size` $\in \{32, 256, 2048\}$.

Daje to 9 treningów. Dla każdego zapisz val MSE po 30 epokach. Zestaw wyniki w tabeli 3 × 3 (`lr` jako wiersze, `batch_size` jako kolumny).

Obserwacje do komentarza:

- Czy większy `batch_size` przyspiesza czy spowalnia zbieżność (w jednostkach epok)?
- Który `lr` jest najlepszy? Czy ten sam dla każdego `batch_size`?
- Czy dla jakiejś kombinacji trening się rozbiega (NaN-y)?

### 2.10 Co model "myśli" o przyszłości?

Model został wytrenowany na przedziale $\tau \in [0, 1]$, który w oryginalnej skali odpowiada zakresowi lat `t_min` do `t_max`. Co stanie się, jeśli zastosujemy go do czasów **poza** tym zakresem?

1. Wygeneruj siatkę lat od `t_max` do `t_max + 10` (gęsto, np. co 10 dni).
2. Przeskaluj do $\tau$ tym samym wzorem co w 2.2 (wartości $\tau$ będą $> 1$).
3. Oblicz $\Phi$ i predykcję modelu.
4. Narysuj razem: dane treningowe + predykcja na pełnym zakresie (wliczając ekstrapolację), zaznacz linią pionową punkt `t_max`.

Co widzisz? Dlaczego model tak się zachowuje poza zakresem? (*Wskazówka:* zerknij na wykres baz z 2.3 — wszystkie centra $\mu_k \in [0, 1]$, więc dla $\tau \gg 1$ wartości $\varphi_k$ są...)

To ważna obserwacja: modele regresji z lokalnymi bazami **nie ekstrapolują** w sensowny sposób. Przewidują dobrze tam, gdzie widziały dane, i "wygasają" poza zakresem.

## 3. Regresja logistyczna — klasyfikacja binarna

W tej sekcji zmieniamy zarówno model, jak i funkcję straty. Model to:

$$\hat{y}(\mathbf{x}, \mathbf{w}) = \sigma(\mathbf{x}^T \mathbf{w}), \qquad \sigma(z) = \frac{1}{1 + e^{-z}},$$

gdzie $\sigma$ to **funkcja sigmoidalna**, a $\hat{y}$ interpretujemy jako prawdopodobieństwo klasy 1. Funkcja straty to **entropia krzyżowa binarna** (BCE) z wykładu 4:

$$L(\mathbf{w}) = -\frac{1}{N} \sum_{n=1}^N \left[ y_n \log \hat{y}_n + (1 - y_n) \log (1 - \hat{y}_n) \right].$$

**Gradient — podajemy wprost** (szkic wyprowadzenia w podręcznikach ML; kluczowy fakt to uproszczenie dzięki strukturze $\sigma$ i BCE):

$$\nabla_{\mathbf{w}} L = \frac{1}{N} X^T (\sigma(X\mathbf{w}) - \mathbf{y}).$$

Dla mini-batcha $B$:

$$\nabla_{\mathbf{w}} L_B = \frac{1}{B} X_B^T (\sigma(X_B \mathbf{w}) - \mathbf{y}_B).$$

Zauważ, że to **ta sama postać** co gradient MSE: $X^T \cdot (\text{predykcja} - \text{etykieta})$. To nie przypadek — dla tej konkretnej pary (funkcja aktywacji, funkcja straty) człony się skracają. Dla innych kombinacji wzór byłby bardziej skomplikowany.

:::{.callout-note}
**Uwaga metodologiczna.** Dla regresji logistycznej na tak małym zbiorze (kilkaset próbek) **nikt w praktyce nie używa GD**. Scikit-learnowy `LogisticRegression` domyślnie używa metody L-BFGS, która zbiega w kilkudziesięciu iteracjach bez konieczności dobierania `lr`. Wybieramy tu GD z powodów dydaktycznych — to ten sam algorytm, który za chwilę będzie napędzał trening sieci neuronowych, gdzie metody drugiego rzędu przestają być praktyczne.
:::

### 3.1 Dane

Użyjemy zbioru **Breast Cancer Wisconsin** (znanego z lab 4 i 5): 569 próbek, 30 cech, 2 klasy (łagodny/złośliwy).

1. Załaduj dane: `from sklearn.datasets import load_breast_cancer`, `data = load_breast_cancer()`.
2. Podziel na train/test 80/20 z `random_state=42` (`sklearn.model_selection.train_test_split`).
3. Znormalizuj cechy używając `StandardScaler` — **dopasuj scaler tylko na train**, transformuj train i test. (GD bez standaryzacji cech zbiega wolno lub wcale — cechy w zbiorze mają rzędy wielkości różne skale.)
4. Do macierzy cech dodaj kolumnę jedynek (bias). Konwertuj na tensory `float32` (cechy) i `float32` (etykiety).
5. Zbuduj `TensorDataset` i `DataLoader` dla zbioru treningowego, `batch_size=32`, `shuffle=True`. Zbiór testowy zostaw jako surowe tensory.

### 3.2 Implementacja sigmoid i BCE

1. Zaimplementuj `sigmoid(z)` ręcznie przez `1 / (1 + torch.exp(-z))` (nie używaj `torch.sigmoid`).
2. Zaimplementuj `bce_loss(y_true, y_pred)`:
   $$\text{BCE} = -\frac{1}{N} \sum_n [y_n \log \hat{y}_n + (1 - y_n) \log(1 - \hat{y}_n)].$$
   
   *Wskazówka dot. stabilności numerycznej:* dla $\hat{y}$ bliskich 0 lub 1, $\log \hat{y}$ dąży do $-\infty$. Zastosuj `torch.clamp(y_pred, eps, 1 - eps)` z małym `eps` (np. $10^{-7}$) przed obliczeniem logarytmu.

### 3.3 Pętla treningowa

Zaimplementuj mini-batch GD analogicznie do sekcji 2.6:

1. Inicjalizuj wagi losowo, np. `torch.randn(n_features + 1) * 0.01`.
2. W każdej epoce iteruj po `DataLoader`ze, licz predykcję, gradient, aktualizuj wagi.
3. Po każdej epoce zapisuj do historii: BCE na train (uśrednione po batchach lub obliczone od razu na całości), BCE na test, accuracy train i test.

Proponowane hiperparametry na start: `lr=0.1`, 100 epok.

*Wskazówka dot. accuracy:* predykcja klasy to `y_pred > 0.5`.

### 3.4 Krzywe uczenia i ewaluacja

1. Narysuj krzywe uczenia:
   - BCE train i test vs epoka (jeden wykres),
   - accuracy train i test vs epoka (drugi wykres).
2. Wypisz końcową accuracy na zbiorze testowym.
3. Opcjonalnie: narysuj macierz pomyłek na zbiorze testowym (`sklearn.metrics.confusion_matrix`).

### 3.5 Sanity check ze scikit-learn

Uruchom `sklearn.linear_model.LogisticRegression` na tych samych przetworzonych danych:

```python
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_scaled, y_train)
```

(Pamiętaj, żeby sklearnowi podać dane **bez** ręcznie doklejonej kolumny jedynek — sklearn dokłada bias sam.)

Porównaj:

1. Accuracy na teście. Powinna być bardzo zbliżona do Twojej implementacji.
2. Wagi (`clf.coef_` i `clf.intercept_`) z wyuczonymi $\mathbf{w}$. Uwaga: sklearn domyślnie stosuje regularyzację L2 z $C=1.0$, więc wagi będą lekko "skurczone" względem Twoich. Porównuj znaki, rzędy wielkości i względne wielkości, a nie dokładne wartości.

## 4. Podsumowanie i zapowiedź wykładu 5

W tym arkuszu wytrenowaliśmy dwa modele za pomocą tego samego algorytmu (mini-batch gradient descent), ale z dwiema różnymi funkcjami straty (MSE, BCE) i dwoma różnymi modelami (liniowy w bazach gaussowskich, regresja logistyczna). Za każdym razem musieliśmy:

1. wyprowadzić analitycznie gradient funkcji straty względem wag,
2. zaprogramować go w postaci wyrażenia macierzowego,
3. wpleść w pętlę treningową.

Dla tych dwóch modeli wzór na gradient miał ładną, wspólną postać $X^T (\text{predykcja} - \text{etykieta})$. Dla bardziej złożonych modeli — a takimi są sieci neuronowe — wyprowadzanie gradientu ręcznie staje się żmudne i łatwe o pomyłkę. Rozważmy na przykład model:

$$\hat{y} = \sigma\!\left(\text{ReLU}(X W_1 + \mathbf{b}_1) W_2 + \mathbf{b}_2\right).$$

Wyprowadzenie $\partial L / \partial W_1$ wymaga kilkukrotnego zastosowania reguły łańcuchowej i sumy iloczynów Hadamarda z pochodnymi funkcji aktywacji. Dla sieci z 10 warstwami i niestandardowymi funkcjami aktywacji — praktycznie niewykonalne.

Na wykładzie 5 poznamy mechanizm **różniczkowania automatycznego** (*autograd*) wbudowany w PyTorch, który sam konstruuje graf obliczeniowy i liczy wszystkie gradienty za nas. Otworzy nam to drogę do trenowania sieci neuronowych praktycznie dowolnej architektury.